In [ ]:
from __future__ import annotations

import operator
import os
import re
from datetime import date, timedelta
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_tavily import TavilySearch
from dotenv import load_dotenv




In [ ]:
# -----------------------------
# 1) Schemas
# -----------------------------
class Task(BaseModel):
    id: int
    title: str

    goal: str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section.",
    )
    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=6,
        description="3–6 concrete, non-overlapping subpoints to cover in this section.",
    )
    target_words: int = Field(..., description="Target word count for this section (120–550).")

    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False


class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks: List[Task]


class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None  # keep if Tavily provides; DO NOT rely on it
    snippet: Optional[str] = None
    source: Optional[str] = None


class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)


class EvidencePack(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)


class DiagramSpec(BaseModel):
    placeholder: str = Field(..., description="e.g. [[DIAGRAM_1]]")
    title: str = Field(..., description="Short caption shown under the diagram.")
    mermaid: str = Field(..., description="Valid Mermaid diagram code (raw, no ``` fences).")


class GlobalDiagramPlan(BaseModel):
    md_with_placeholders: str
    diagrams: List[DiagramSpec] = Field(default_factory=list)

In [ ]:
class State(TypedDict):
    topic: str

    # routing / research
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int, str]], operator.add]  # (task_id, section_md)

    # reducer/diagrams
    merged_md: str
    md_with_placeholders: str
    diagram_specs: List[dict]

    final: str


# -----------------------------
# 2) LLM  (Groq)
# -----------------------------
from langchain_groq import ChatGroq  # pip install langchain-groq

load_dotenv()

# Groq free tier. Swap `model` for any tool-calling model on
# https://console.groq.com/docs/models (e.g. "openai/gpt-oss-120b").
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

In [ ]:
# -----------------------------
# 3) Router (decide upfront)
# -----------------------------
ROUTER_SYSTEM = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 3–10 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- If user asked for "last week/this week/latest", reflect that constraint IN THE QUERIES.
"""

def router_node(state: State) -> dict:
    
    topic = state["topic"]
    decider = llm.with_structured_output(RouterDecision)
    decision = decider.invoke(
        [
            SystemMessage(content=ROUTER_SYSTEM),
            HumanMessage(content=f"Topic: {topic}"),
        ]
    )

    return {
        "needs_research": decision.needs_research,
        "mode": decision.mode,
        "queries": decision.queries,
    }

def route_next(state: State) -> str:
    return "research" if state["needs_research"] else "orchestrator"


In [ ]:
# -----------------------------
# 4) Research (Tavily) 
# -----------------------------
def _tavily_search(query: str, max_results: int = 5) -> List[dict]:
    """
    Uses TavilySearch if installed and TAVILY_API_KEY is set.
    Returns list of dict with common fields. Note: published date is often missing.
    """
    tool = TavilySearch(max_results=max_results)
    response = tool.invoke({"query": query})

    # TavilySearch.invoke returns a dict like:
    # {"query": ..., "results": [...], "images": [...], ...}
    # not a bare list — pull the actual results list out.
    if isinstance(response, dict):
        items = response.get("results") or []
    elif isinstance(response, list):
        items = response
    else:
        items = []

    normalized: List[dict] = []
    for r in items:
        if not isinstance(r, dict):
            continue
        normalized.append(
            {
                "title": r.get("title") or "",
                "url": r.get("url") or "",
                "snippet": r.get("content") or r.get("snippet") or "",
                "published_at": r.get("published_date") or r.get("published_at"),
                "source": r.get("source"),
            }
        )
    return normalized


RESEARCH_SYSTEM = """You are a research synthesizer for technical writing.

Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources (company blogs, docs, reputable outlets).
- If a published date is explicitly present in the result payload, keep it as YYYY-MM-DD.
  If missing or unclear, set published_at=null. Do NOT guess.
- Keep snippets short.
- Deduplicate by URL.
"""

def research_node(state: State) -> dict:

    # take the first 10 queries from state
    queries = (state.get("queries", []) or [])
    max_results = 6

    raw_results: List[dict] = []

    for q in queries:
        raw_results.extend(_tavily_search(q, max_results=max_results))

    if not raw_results:
        return {"evidence": []}

    extractor = llm.with_structured_output(EvidencePack)
    pack = extractor.invoke(
        [
            SystemMessage(content=RESEARCH_SYSTEM),
            HumanMessage(content=f"Raw results:\n{raw_results}"),
        ]
    )

    # Deduplicate by URL
    dedup = {}
    for e in pack.evidence:
        if e.url:
            dedup[e.url] = e

    return {"evidence": list(dedup.values())}


In [ ]:
# -----------------------------
# 5) Orchestrator (Plan)
# -----------------------------
ORCH_SYSTEM = """You are a senior technical writer and developer advocate.
Your job is to produce a highly actionable outline for a technical blog post.

Hard requirements:
- Create 5–9 sections (tasks) suitable for the topic and audience.
- Each task must include:
  1) goal (1 sentence)
  2) 3–6 bullets that are concrete, specific, and non-overlapping
  3) target word count (120–550)

Quality bar:
- Assume the reader is a developer; use correct terminology.
- Bullets must be actionable: build/compare/measure/verify/debug.
- Ensure the overall plan includes at least 2 of these somewhere:
  * minimal code sketch / MWE (set requires_code=True for that section)
  * edge cases / failure modes
  * performance/cost considerations
  * security/privacy considerations (if relevant)
  * debugging/observability tips

Grounding rules:
- Mode closed_book: keep it evergreen; do not depend on evidence.
- Mode hybrid:
  - Use evidence for up-to-date examples (models/tools/releases) in bullets.
  - Mark sections using fresh info as requires_research=True and requires_citations=True.
- Mode open_book:
  - Set blog_kind = "news_roundup".
  - Every section is about summarizing events + implications.
  - DO NOT include tutorial/how-to sections unless user explicitly asked for that.
  - If evidence is empty or insufficient, create a plan that transparently says "insufficient sources"
    and includes only what can be supported.

Output must strictly match the Plan schema.
"""

def orchestrator_node(state: State) -> dict:
    planner = llm.with_structured_output(Plan)

    evidence = state.get("evidence", [])
    mode = state.get("mode", "closed_book")

    plan = planner.invoke(
        [
            SystemMessage(content=ORCH_SYSTEM),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Mode: {mode}\n\n"
                    f"Evidence (ONLY use for fresh claims; may be empty):\n"
                    f"{[e.model_dump() for e in evidence][:16]}"
                )
            ),
        ]
    )

    return {"plan": plan}

# -----------------------------
# 6) Fanout
# -----------------------------
def fanout(state: State):
    return [
        Send(
            "worker",
            {
                "task": task.model_dump(),
                "topic": state["topic"],
                "mode": state["mode"],
                "plan": state["plan"].model_dump(),
                "evidence": [e.model_dump() for e in state.get("evidence", [])],
            },
        )
        for task in state["plan"].tasks
    ]



In [ ]:
# -----------------------------
# 7) Worker (write one section)
# -----------------------------
WORKER_SYSTEM = """You are a senior technical writer and developer advocate.
Write ONE section of a technical blog post in Markdown.

Hard constraints:
- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).
- Stay close to Target words (±15%).
- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).
- Start with a '## <Section Title>' heading.

Scope guard:
- If blog_kind == "news_roundup": do NOT turn this into a tutorial/how-to guide.
  Do NOT teach web scraping, RSS, automation, or "how to fetch news" unless bullets explicitly ask for it.
  Focus on summarizing events and implications.

Grounding policy:
- If mode == open_book:
  - Do NOT introduce any specific event/company/model/funding/policy claim unless it is supported by provided Evidence URLs.
  - For each event claim, attach a source as a Markdown link: ([Source](URL)).
  - Only use URLs provided in Evidence. If not supported, write: "Not found in provided sources."
- If requires_citations == true:
  - For outside-world claims, cite Evidence URLs the same way.
- Evergreen reasoning is OK without citations unless requires_citations is true.

Code:
- If requires_code == true, include at least one minimal, correct code snippet relevant to the bullets.

Style:
- Short paragraphs, bullets where helpful, code fences for code.
- Avoid fluff/marketing. Be precise and implementation-oriented.
"""

def worker_node(payload: dict) -> dict:
    
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = [EvidenceItem(**e) for e in payload.get("evidence", [])]
    topic = payload["topic"]
    mode = payload.get("mode", "closed_book")

    bullets_text = "\n- " + "\n- ".join(task.bullets)

    evidence_text = ""
    if evidence:
        evidence_text = "\n".join(
            f"- {e.title} | {e.url} | {e.published_at or 'date:unknown'}".strip()
            for e in evidence[:20]
        )

    section_md = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog title: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Constraints: {plan.constraints}\n"
                    f"Topic: {topic}\n"
                    f"Mode: {mode}\n\n"
                    f"Section title: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Tags: {task.tags}\n"
                    f"requires_research: {task.requires_research}\n"
                    f"requires_citations: {task.requires_citations}\n"
                    f"requires_code: {task.requires_code}\n"
                    f"Bullets:{bullets_text}\n\n"
                    f"Evidence (ONLY use these URLs when citing):\n{evidence_text}\n"
                )
            ),
        ]
    ).content.strip()

    return {"sections": [(task.id, section_md)]}


In [ ]:
# ============================================================
# 8) ReducerWithDiagrams (subgraph)
#    merge_content -> decide_images -> generate_and_place_images
#    (diagrams are inline Mermaid — free, deterministic; validated via kroki)
# ============================================================
import time
import requests


def merge_content(state: State) -> dict:

    plan = state["plan"]

    ordered_sections = [md for _, md in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered_sections).strip()
    merged_md = f"# {plan.blog_title}\n\n{body}\n"
    return {"merged_md": merged_md}


DECIDE_DIAGRAMS_SYSTEM = """You are an expert technical editor.
Decide whether diagrams would materially help readers understand THIS blog,
then express each one as a Mermaid diagram.

Rules:
- Max 3 diagrams total. Add a diagram ONLY where it clarifies a flow,
  architecture, process, or relationship that prose/code handles poorly.
- Insert placeholders exactly: [[DIAGRAM_1]], [[DIAGRAM_2]], [[DIAGRAM_3]],
  each on its own line at the most relevant spot in the markdown.
- If no diagrams help: md_with_placeholders must EQUAL the input and diagrams=[].

Mermaid rules (CRITICAL — output must be VALID Mermaid that renders on GitHub):
- Pick the fitting type: flowchart (`graph TD` / `graph LR`), `sequenceDiagram`,
  `stateDiagram-v2`, `classDiagram`, or `erDiagram`.
- Put ONLY raw Mermaid code in the `mermaid` field. Do NOT wrap it in ``` fences.
- Put EACH statement on its own line. Do NOT cram the whole graph onto one line
  and do NOT use ';' as a separator.
- For a labeled edge use EXACTLY this form:  A -->|"label"| B
  The label is closed by a single '|' followed by the target node.
  NEVER write a trailing '>' after the label (i.e. `A -->|label|> B` is WRONG).
- Keep node labels short. Wrap any label with spaces/special chars in double
  quotes, e.g.  A["Scaled dot-product"] --> B["Softmax"].
- Avoid raw parentheses or unquoted punctuation inside labels.
- Prefer 4–12 nodes; keep it readable, not exhaustive.

Example of a valid diagram:
graph LR
    A["Input"] -->|"Query"| B["Q vector"]
    A -->|"Key"| C["K vector"]
    B --> D["Attention scores"]
    C --> D

Return strictly GlobalDiagramPlan.
"""


def decide_images(state: State) -> dict:

    planner = llm.with_structured_output(GlobalDiagramPlan)
    merged_md = state["merged_md"]
    plan = state["plan"]
    assert plan is not None

    diagram_plan = planner.invoke(
        [
            SystemMessage(content=DECIDE_DIAGRAMS_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Topic: {state['topic']}\n\n"
                    "Insert placeholders + propose Mermaid diagrams.\n\n"
                    f"{merged_md}"
                )
            ),
        ]
    )

    return {
        "md_with_placeholders": diagram_plan.md_with_placeholders,
        "diagram_specs": [d.model_dump() for d in diagram_plan.diagrams],
    }


def _clean_mermaid(code: str) -> str:
    """Normalize LLM-produced Mermaid into valid, renderable code."""
    code = (code or "").strip()

    # 1) Strip accidental ``` fences / language tags.
    if code.startswith("```"):
        lines = code.splitlines()
        if lines and lines[0].startswith("```"):      # drop opening ``` / ```mermaid
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):  # drop closing ```
            lines = lines[:-1]
        code = "\n".join(lines).strip()

    # 2) Repair a common LLM mistake: labeled edges written as `-->|text|>`.
    #    Valid Mermaid is `-->|text|` (no trailing '>'). Verified against a
    #    real renderer: the trailing '>' returns a hard HTTP 400.
    code = code.replace("|>", "|")

    # 3) Split a ';'-separated single line onto its own lines. Semicolons are
    #    valid Mermaid, but the one-line form is unreadable and some renderers
    #    dislike it; the model also ignores the "no semicolons" instruction.
    if ";" in code:
        parts = [p.strip() for p in code.split(";") if p.strip()]
        if parts:
            header, *rest = parts
            code = header + "\n" + "\n".join("    " + r for r in rest)

    return code


def _mermaid_ok(src: str) -> bool:
    """True if the Mermaid renders. Validate via kroki; only a definitive HTTP
    400 counts as invalid. kroki's 500s are flaky, so retry to reach a real
    200/400, and fail OPEN if we never do (a hiccup must not strip good diagrams).
    """
    for _ in range(3):
        try:
            r = requests.post(
                "https://kroki.io/mermaid/svg",
                data=src.encode("utf-8"),
                timeout=30,
            )
            if r.status_code == 400:   # definitive syntax error
                return False
            if r.status_code == 200:   # definitely renders
                return True
            # 5xx -> transient, retry
        except Exception:
            pass
        time.sleep(1)
    return True  # no definitive answer -> keep it (fail-open)


def generate_and_place_images(state: State) -> dict:

    plan = state["plan"]
    assert plan is not None

    md = state.get("md_with_placeholders") or state["merged_md"]
    diagram_specs = state.get("diagram_specs", []) or []

    for spec in diagram_specs:
        placeholder = spec["placeholder"]
        mermaid = _clean_mermaid(spec.get("mermaid", ""))
        title = (spec.get("title") or "").strip()

        # Empty diagram -> just remove the placeholder, keep doc clean.
        if not mermaid:
            md = md.replace(placeholder, "")
            continue

        # Never ship a syntax-broken diagram.
        if not _mermaid_ok(mermaid):
            print(f"⚠️  Dropping invalid Mermaid at {placeholder}")
            md = md.replace(placeholder, "")
            continue

        block = f"```mermaid\n{mermaid}\n```"
        if title:
            block += f"\n*{title}*"
        md = md.replace(placeholder, block)

    filename = f"{plan.blog_title}.md"
    Path(filename).write_text(md, encoding="utf-8")
    return {"final": md}

In [ ]:
# build reducer subgraph
reducer_graph = StateGraph(State)
reducer_graph.add_node("merge_content", merge_content)
reducer_graph.add_node("decide_images", decide_images)
reducer_graph.add_node("generate_and_place_images", generate_and_place_images)
reducer_graph.add_edge(START, "merge_content")
reducer_graph.add_edge("merge_content", "decide_images")
reducer_graph.add_edge("decide_images", "generate_and_place_images")
reducer_graph.add_edge("generate_and_place_images", END)
reducer_subgraph = reducer_graph.compile()

reducer_subgraph


In [ ]:
# -----------------------------
# 9) Build main graph
# -----------------------------
g = StateGraph(State)
g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_subgraph)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")

g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()
app


In [ ]:
# -----------------------------
# 10) Runner
# -----------------------------
def run(topic: str, as_of: Optional[str] = None):
    if as_of is None:
        as_of = date.today().isoformat()

    out = app.invoke(
        {
            "topic": topic,
            "mode": "",
            "needs_research": False,
            "queries": [],
            "evidence": [],
            "plan": None,
            "as_of": as_of,
            "recency_days": 7,
            "sections": [],
            "merged_md": "",
            "md_with_placeholders": "",
            "diagram_specs": [],
            "final": "",
        }
    )

    return out

In [ ]:
out=run("Transformer Architecture in AI")

In [ ]:
Path("../outputs/blog8.md").write_text(out["final"], encoding="utf-8")